# Algoritmo de la Colonia de Hormigas

**Alumno:** Alejandro Matías Ponce Ocampo  
**Materia:** Procesamiento Inteligente de Datos  
**Universidad Anáhuac Mayab**


## El problema del viajero

### Descripción general

El problema del viajero (TSP) consiste en encontrar la ruta más corta que visite un conjunto de ciudades exactamente una vez y regrese al punto de partida. Es un problema clásico de optimización combinatoria y es NP-duro, es decir, no existe un algoritmo que lo resuelva de forma exacta en tiempo polinómico.

### Objetivos

* Minimizar la distancia total del recorrido.
* Encontrar soluciones buenas en tiempo razonable para instancias grandes.
* Aplicarlo a problemas reales como rutas de reparto, fabricación de circuitos o secuenciación de ADN.

### Limitaciones

* Es NP-duro, así que los métodos exactos solo sirven para pocas ciudades.
* Las metaheurísticas como ACO dan buenas soluciones pero no garantizan que sean óptimas.
* Los resultados dependen de los parámetros elegidos (alfa, beta, evaporación, iteraciones).

### Trabajos previos

* 1930: Karl Menger formula el problema matemáticamente.
* 1954: Dantzig, Fulkerson y Johnson resuelven una instancia de 49 ciudades.
* 1972: Karp demuestra que el TSP es NP-completo.
* 1991: Marco Dorigo propone el Ant System (ACO) en su tesis doctoral.
* Actualidad: solvers como Concorde y Google OR-Tools resuelven instancias con miles de ciudades.


## Código del libro

Nota: se cambió `sys.maxint` por `sys.maxsize` porque el primero es de Python 2 y ya no existe en Python 3.


### Importaciones


In [1]:
import random, sys, math

# Nota: en lugar de matrices se usan listas de listas
print("Módulos importados correctamente")

Módulos importados correctamente


### Matriz de distancias


In [2]:
# Genera una matriz de distancias de nCiudades x nCiudades
def matrizDistancias(nCiud, distanciaMaxima):
    matriz = [[0 for i in range(nCiud)] for j in range(nCiud)]

    for i in range(nCiud):
        for j in range(i):
            matriz[i][j] = distanciaMaxima*random.random()
            matriz[j][i] = matriz[i][j]

    return matriz

# Prueba: matriz de 5 ciudades con distancia máxima 10
matrizPrueba = matrizDistancias(5, 10)
print("Matriz de distancias de prueba (5x5):")
for fila in matrizPrueba:
    print([round(x, 2) for x in fila])

Matriz de distancias de prueba (5x5):
[0, 7.87, 8.29, 8.9, 0.89]
[7.87, 0, 2.22, 8.32, 8.05]
[8.29, 2.22, 0, 6.07, 7.34]
[8.9, 8.32, 6.07, 0, 8.58]
[0.89, 8.05, 7.34, 8.58, 0]


### Elegir la siguiente ciudad


In [3]:
# Elige un paso de una hormiga, teniendo en cuenta las distancias
# y las feromonas y descartando las ciudades ya visitadas.
def eligeCiudad(dists, ferom, visitadas):
    # Se calcula la tabla de pesos de cada ciudad
    listaPesos  = []
    disponibles = []
    actual      = visitadas[-1]

    # Influencia de cada valor (alfa: feromonas; beta: distancias)
    alfa = 1.0
    beta = 0.5

    # El parámetro beta (peso de las distancias) es 0.5, alfa=1.0
    for i in range(len(dists)):
        if i not in visitadas:
            fer  = math.pow((1.0 + ferom[actual][i]), alfa)
            peso = math.pow(1.0/dists[actual][i], beta) * fer
            disponibles.append(i)
            listaPesos.append(peso)

    # Se elige aleatoriamente una de las ciudades disponibles,
    # teniendo en cuenta su peso relativo.
    valor     = random.random() * sum(listaPesos)
    acumulado = 0.0
    i         = -1
    while valor > acumulado:
        i         += 1
        acumulado += listaPesos[i]

    return disponibles[i]

# Prueba: estando en la ciudad 0, elegir la siguiente con feromonas vacías
feromonasVacias = [[0]*5 for _ in range(5)]
siguiente = eligeCiudad(matrizPrueba, feromonasVacias, [0])
print("Desde la ciudad 0, la hormiga elige ir a la ciudad:", siguiente)

Desde la ciudad 0, la hormiga elige ir a la ciudad: 2


### Construir un camino completo


In [4]:
# Genera una "hormiga", que elegirá un camino teniendo en cuenta
# las distancias y los rastros de feromonas. Devuelve una tupla
# con el camino y su longitud.
def eligeCamino(distancias, feromonas):
    # La ciudad inicial siempre es la 0
    camino     = [0]
    longCamino = 0

    # Elegir cada paso según la distancia y las feromonas
    while len(camino) < len(distancias):
        ciudad      = eligeCiudad(distancias, feromonas, camino)
        longCamino += distancias[camino[-1]][ciudad]
        camino.append(ciudad)

    # Para terminar hay que volver a la ciudad de origen (0)
    longCamino += distancias[camino[-1]][0]
    camino.append(0)

    return (camino, longCamino)

# Prueba: generar un camino completo sobre la matriz de 5 ciudades
(caminoPrueba, longPrueba) = eligeCamino(matrizPrueba, feromonasVacias)
print("Camino generado:", caminoPrueba)
print("Longitud del camino:", round(longPrueba, 2))

Camino generado: [0, 3, 2, 4, 1, 0]
Longitud del camino: 38.23


### Actualizar y evaporar feromonas


In [5]:
# Actualiza la matriz de feromonas siguiendo el camino recibido
def rastroFeromonas(feromonas, camino, dosis):
    for i in range(len(camino) - 1):
        feromonas[camino[i]][camino[i+1]] += dosis

# Evapora todas las feromonas multiplicándolas por una constante
# = 0.9 (en otras palabras, el coeficiente de evaporación es 0.1)
def evaporaFeromonas(feromonas):
    for lista in feromonas:
        for i in range(len(lista)):
            lista[i] *= 0.9

# Prueba: dejar rastro del camino anterior con dosis 1.0 y luego evaporar
feromonasPrueba = [[0]*5 for _ in range(5)]
rastroFeromonas(feromonasPrueba, caminoPrueba, 1.0)
print("Feromonas después del rastro (dosis=1.0):")
for fila in feromonasPrueba:
    print([round(x, 2) for x in fila])

evaporaFeromonas(feromonasPrueba)
print("\nFeromonas después de evaporar (x 0.9):")
for fila in feromonasPrueba:
    print([round(x, 2) for x in fila])

Feromonas después del rastro (dosis=1.0):
[0, 0, 0, 1.0, 0]
[1.0, 0, 0, 0, 0]
[0, 0, 0, 0, 1.0]
[0, 0, 1.0, 0, 0]
[0, 1.0, 0, 0, 0]

Feromonas después de evaporar (x 0.9):
[0.0, 0.0, 0.0, 0.9, 0.0]
[0.9, 0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.9]
[0.0, 0.0, 0.9, 0.0, 0.0]
[0.0, 0.9, 0.0, 0.0, 0.0]


### Algoritmo principal y prueba final


In [6]:
# Resuelve el problema del viajante de comercio mediante el
# algoritmo de la colonia de hormigas. Recibe una matriz de
# distancias y devuelve una tupla con el mejor camino que ha
# obtenido (lista de índices) y su longitud
def hormigas(distancias, iteraciones, distMedia):
    # Primero se crea una matriz de feromonas vacía
    n         = len(distancias)
    feromonas = [[0 for i in range(n)] for j in range(n)]

    # El mejor camino y su longitud (inicialmente "infinita")
    mejorCamino     = []
    longMejorCamino = sys.maxsize  # En el libro aparece como sys.maxint (Python 2)

    # En cada iteración se genera una hormiga, que elige un camino,
    # y si es mejor que el mejor que teníamos, deja su rastro de
    # feromonas (mayor cuanto más corto sea el camino)
    for iter in range(iteraciones):
        (camino, longCamino) = eligeCamino(distancias, feromonas)

        if longCamino <= longMejorCamino:
            mejorCamino     = camino
            longMejorCamino = longCamino

        rastroFeromonas(feromonas, camino, distMedia/longCamino)

        # En cualquier caso, las feromonas se van evaporando
        evaporaFeromonas(feromonas)

    # Se devuelve el mejor camino que se haya encontrado
    return (mejorCamino, longMejorCamino)

# Generación de una matriz de prueba
numCiudades     = 10
distanciaMaxima = 10
ciudades        = matrizDistancias(numCiudades, distanciaMaxima)

# Obtención del mejor camino
iteraciones = 1000
distMedia   = numCiudades*distanciaMaxima/2
(camino, longCamino) = hormigas(ciudades, iteraciones, distMedia)
print("Camino:", camino)
print("Longitud del camino:", longCamino)

Camino: [0, 1, 8, 4, 6, 5, 7, 9, 3, 2, 0]
Longitud del camino: 18.04551839178814
